Create a data pipeline that gets us playoff information regarding how far teams went into the playoffs in boolean format while using a data source from Basketball Reference

# S03 — Postseason Outcomes

Target grain:
One row = one NBA team × season

Goal:
Transform historical playoff-series data into team-season postseason
milestones and final postseason outcomes.

In [2]:
import pandas as pd
import ssl
import certifi
import urllib.request
import io

url = "https://www.basketball-reference.com/playoffs/series.html"

context = ssl.create_default_context(cafile=certifi.where())

with urllib.request.urlopen(url, context=context) as response:
    html = response.read()

tables = pd.read_html(io.BytesIO(html))

In [3]:
playoff_info = tables[0].copy()

playoff_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067 entries, 0 to 1066
Data columns (total 13 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   (Unnamed: 0_level_0, Yr)                    1008 non-null   str    
 1   (Unnamed: 1_level_0, Lg)                    1008 non-null   str    
 2   (Unnamed: 2_level_0, Series)                1008 non-null   str    
 3   (Unnamed: 3_level_0, Unnamed: 3_level_1)    979 non-null    str    
 4   (Unnamed: 4_level_0, Unnamed: 4_level_1)    0 non-null      float64
 5   (Winner, Team)                              1008 non-null   str    
 6   (Winner, W)                                 1008 non-null   str    
 7   (Unnamed: 7_level_0, Unnamed: 7_level_1)    0 non-null      float64
 8   (Loser, Team)                               1008 non-null   str    
 9   (Loser, W)                                  1008 non-null   str    
 10  (Unnamed: 10_level_0, U

In [4]:
playoff_info.shape

(1067, 13)

In [5]:
playoff_info.columns

MultiIndex([( 'Unnamed: 0_level_0',                  'Yr'),
            ( 'Unnamed: 1_level_0',                  'Lg'),
            ( 'Unnamed: 2_level_0',              'Series'),
            ( 'Unnamed: 3_level_0',  'Unnamed: 3_level_1'),
            ( 'Unnamed: 4_level_0',  'Unnamed: 4_level_1'),
            (             'Winner',                'Team'),
            (             'Winner',                   'W'),
            ( 'Unnamed: 7_level_0',  'Unnamed: 7_level_1'),
            (              'Loser',                'Team'),
            (              'Loser',                   'W'),
            ('Unnamed: 10_level_0', 'Unnamed: 10_level_1'),
            ('Unnamed: 11_level_0',            'Favorite'),
            ('Unnamed: 12_level_0',            'Underdog')],
           )

In [6]:
new_columns = [
    "year",
    "league",
    "playoff round",
    "dates",
    "separator_1",
    "winning team",
    "winning team wins",
    "separator_2",
    "losing team",
    "losing team wins",
    "separator_3",
    "favorite",
    "underdog"
]

len(new_columns)

13

In [7]:
playoff_info.columns = new_columns

print(playoff_info.info())

<class 'pandas.DataFrame'>
RangeIndex: 1067 entries, 0 to 1066
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               1008 non-null   str    
 1   league             1008 non-null   str    
 2   playoff round      1008 non-null   str    
 3   dates              979 non-null    str    
 4   separator_1        0 non-null      float64
 5   winning team       1008 non-null   str    
 6   winning team wins  1008 non-null   str    
 7   separator_2        0 non-null      float64
 8   losing team        1008 non-null   str    
 9   losing team wins   1008 non-null   str    
 10  separator_3        0 non-null      float64
 11  favorite           608 non-null    str    
 12  underdog           608 non-null    str    
dtypes: float64(3), str(10)
memory usage: 108.5 KB
None


In [8]:
playoff_info.head()

,year,league,playoff round,dates,separator_1,winning team,winning team wins,separator_2,losing team,losing team wins,separator_3,favorite,underdog
0,2026,NBA,Eastern Conf First Round,"Apr 19 - May 3, 2026",NaN,Detroit Pistons (1),4,NaN,Orlando Magic (8),3,NaN,DET (-600),ORL (+450)
1,2026,NBA,Eastern Conf First Round,"Apr 18 - May 3, 2026",NaN,Cleveland Cavaliers (4),4,NaN,Toronto Raptors (5),3,NaN,CLE (-550),TOR (+425)
2,2026,NBA,Eastern Conf First Round,"Apr 18 - Apr 30, 2026",NaN,New York Knicks (3),4,NaN,Atlanta Hawks (6),2,NaN,NYK (-300),ATL (+250)
3,2026,NBA,Eastern Conf First Round,"Apr 19 - May 2, 2026",NaN,Philadelphia 76ers (7),4,NaN,Boston Celtics (2),3,NaN,BOS (-1300),PHI (+800)
4,2026,NBA,Western Conf First Round,"Apr 19 - Apr 27, 2026",NaN,Oklahoma City Thunder (1),4,NaN,Phoenix Suns (8),0,NaN,OKC (-6000),PHO (+2200)


In [9]:
playoff_info_no_separators = playoff_info.drop(columns=["separator_1", "separator_2", "separator_3", "favorite", "underdog"])

playoff_info_no_separators.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067 entries, 0 to 1066
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   year               1008 non-null   str  
 1   league             1008 non-null   str  
 2   playoff round      1008 non-null   str  
 3   dates              979 non-null    str  
 4   winning team       1008 non-null   str  
 5   winning team wins  1008 non-null   str  
 6   losing team        1008 non-null   str  
 7   losing team wins   1008 non-null   str  
dtypes: str(8)
memory usage: 66.8 KB


In [10]:
playoff_info_no_separators.isna().sum()

year                 59
league               59
playoff round        59
dates                88
winning team         59
winning team wins    59
losing team          59
losing team wins     59
dtype: int64

In [11]:
playoff_info_no_separators.isna()

,year,league,playoff round,dates,winning team,winning team wins,losing team,losing team wins
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
1062,False,False,False,False,False,False,False,False
1063,False,False,False,False,False,False,False,False
1064,False,False,False,False,False,False,False,False
1065,False,False,False,False,False,False,False,False


In [12]:
playoff_info_collapsed = playoff_info_no_separators[playoff_info_no_separators["playoff round"].notna()]

In [13]:
playoff_info_collapsed.info()

<class 'pandas.DataFrame'>
Index: 1008 entries, 0 to 1066
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   year               1008 non-null   str  
 1   league             1008 non-null   str  
 2   playoff round      1008 non-null   str  
 3   dates              979 non-null    str  
 4   winning team       1008 non-null   str  
 5   winning team wins  1008 non-null   str  
 6   losing team        1008 non-null   str  
 7   losing team wins   1008 non-null   str  
dtypes: str(8)
memory usage: 70.9 KB


In [14]:
playoff_info_collapsed[playoff_info_collapsed["dates"].isna()]

,year,league,playoff round,dates,winning team,winning team wins,losing team,losing team wins
63,Yr,Lg,Series,NaN,Team,W,Team,W
111,Yr,Lg,Series,NaN,Team,W,Team,W
159,Yr,Lg,Series,NaN,Team,W,Team,W
207,Yr,Lg,Series,NaN,Team,W,Team,W
255,Yr,Lg,Series,NaN,Team,W,Team,W
303,Yr,Lg,Series,NaN,Team,W,Team,W
351,Yr,Lg,Series,NaN,Team,W,Team,W
399,Yr,Lg,Series,NaN,Team,W,Team,W
447,Yr,Lg,Series,NaN,Team,W,Team,W
495,Yr,Lg,Series,NaN,Team,W,Team,W


In [15]:
playoff_info_full = playoff_info_collapsed[playoff_info_collapsed["dates"].notna()]

In [16]:
playoff_info_full.info()

<class 'pandas.DataFrame'>
Index: 979 entries, 0 to 1066
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   year               979 non-null    str  
 1   league             979 non-null    str  
 2   playoff round      979 non-null    str  
 3   dates              979 non-null    str  
 4   winning team       979 non-null    str  
 5   winning team wins  979 non-null    str  
 6   losing team        979 non-null    str  
 7   losing team wins   979 non-null    str  
dtypes: str(8)
memory usage: 68.8 KB


In [17]:
playoff_info_full.head()

,year,league,playoff round,dates,winning team,winning team wins,losing team,losing team wins
0,2026,NBA,Eastern Conf First Round,"Apr 19 - May 3, 2026",Detroit Pistons (1),4,Orlando Magic (8),3
1,2026,NBA,Eastern Conf First Round,"Apr 18 - May 3, 2026",Cleveland Cavaliers (4),4,Toronto Raptors (5),3
2,2026,NBA,Eastern Conf First Round,"Apr 18 - Apr 30, 2026",New York Knicks (3),4,Atlanta Hawks (6),2
3,2026,NBA,Eastern Conf First Round,"Apr 19 - May 2, 2026",Philadelphia 76ers (7),4,Boston Celtics (2),3
4,2026,NBA,Western Conf First Round,"Apr 19 - Apr 27, 2026",Oklahoma City Thunder (1),4,Phoenix Suns (8),0


In [18]:
playoff_info_full["playoff round"].value_counts()

playoff round
Eastern Conf First Round          188
Western Conf First Round          188
Western Conf Semifinals           112
Eastern Conf Semifinals           112
Finals                             89
Western Conf Finals                56
Eastern Conf Finals                56
Eastern Div Semifinals             46
Western Div Semifinals             46
Eastern Div Finals                 30
Western Div Finals                 30
Semifinals                          7
Western Div Tiebreak                6
Quarterfinals                       4
Central Div Semifinals              2
First Round                         1
Eastern Div Tiebreak                1
Eastern Div 3rd Place Tiebreak      1
Western Div 2nd Place Tiebreak      1
Central Div 1st Place Tiebreak      1
Central Div 3rd Place Tiebreak      1
Central Div Finals                  1
Name: count, dtype: int64

In [19]:
playoff_info_full["year"] = playoff_info_full["year"].astype(int)

post_2000_playoff_info = (playoff_info_full[playoff_info_full["year"] >= 2001])

post_2000_playoff_info.info()

post_2000_playoff_info["playoff round"].value_counts()

<class 'pandas.DataFrame'>
Index: 390 entries, 0 to 414
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   year               390 non-null    int64
 1   league             390 non-null    str  
 2   playoff round      390 non-null    str  
 3   dates              390 non-null    str  
 4   winning team       390 non-null    str  
 5   winning team wins  390 non-null    str  
 6   losing team        390 non-null    str  
 7   losing team wins   390 non-null    str  
dtypes: int64(1), str(7)
memory usage: 27.4 KB


playoff round
Eastern Conf First Round    104
Western Conf First Round    104
Western Conf Semifinals      52
Eastern Conf Semifinals      52
Western Conf Finals          26
Eastern Conf Finals          26
Finals                       26
Name: count, dtype: int64

In [20]:
post_2000_playoff_info.head()

,year,league,playoff round,dates,winning team,winning team wins,losing team,losing team wins
0,2026,NBA,Eastern Conf First Round,"Apr 19 - May 3, 2026",Detroit Pistons (1),4,Orlando Magic (8),3
1,2026,NBA,Eastern Conf First Round,"Apr 18 - May 3, 2026",Cleveland Cavaliers (4),4,Toronto Raptors (5),3
2,2026,NBA,Eastern Conf First Round,"Apr 18 - Apr 30, 2026",New York Knicks (3),4,Atlanta Hawks (6),2
3,2026,NBA,Eastern Conf First Round,"Apr 19 - May 2, 2026",Philadelphia 76ers (7),4,Boston Celtics (2),3
4,2026,NBA,Western Conf First Round,"Apr 19 - Apr 27, 2026",Oklahoma City Thunder (1),4,Phoenix Suns (8),0


In [21]:
playoff_shrink = post_2000_playoff_info.drop(columns = ["league", "dates"])

In [22]:
post_2000_playoff_info["season"] = (
    (post_2000_playoff_info["year"] - 1).astype(str)
    + "-"
    + post_2000_playoff_info["year"].astype(str).str[-4:]
)

In [23]:
playoff_shrink["season"] = post_2000_playoff_info["season"]

In [24]:
playoff_shrink.head()

,year,playoff round,winning team,winning team wins,losing team,losing team wins,season
0,2026,Eastern Conf First Round,Detroit Pistons (1),4,Orlando Magic (8),3,2025-2026
1,2026,Eastern Conf First Round,Cleveland Cavaliers (4),4,Toronto Raptors (5),3,2025-2026
2,2026,Eastern Conf First Round,New York Knicks (3),4,Atlanta Hawks (6),2,2025-2026
3,2026,Eastern Conf First Round,Philadelphia 76ers (7),4,Boston Celtics (2),3,2025-2026
4,2026,Western Conf First Round,Oklahoma City Thunder (1),4,Phoenix Suns (8),0,2025-2026


In [25]:
playoff_shrink["winning seed"] = (
    playoff_shrink["winning team"]
    .str.extract(r"\((\d+)\)$")
)

playoff_shrink["losing seed"] = (playoff_shrink["losing team"].str.extract(r"\((\d+)\)$"))

playoff_shrink["winning team"] = (
    playoff_shrink["winning team"]
    .str.replace(r"\s*\(\d+\)$", "", regex=True)
)

playoff_shrink["losing team"] = (
    playoff_shrink["losing team"]
    .str.replace(r"\s*\(\d+\)$", "", regex=True)
)

playoff_shrink.head()

,year,playoff round,winning team,winning team wins,losing team,losing team wins,season,winning seed,losing seed
0,2026,Eastern Conf First Round,Detroit Pistons,4,Orlando Magic,3,2025-2026,1,8
1,2026,Eastern Conf First Round,Cleveland Cavaliers,4,Toronto Raptors,3,2025-2026,4,5
2,2026,Eastern Conf First Round,New York Knicks,4,Atlanta Hawks,2,2025-2026,3,6
3,2026,Eastern Conf First Round,Philadelphia 76ers,4,Boston Celtics,3,2025-2026,7,2
4,2026,Western Conf First Round,Oklahoma City Thunder,4,Phoenix Suns,0,2025-2026,1,8


In [26]:
playoff_winners = playoff_shrink[
    [
        "season",
        "year",
        "playoff round",
        "winning team",
        "winning seed",
        "winning team wins"
    ]
].copy()

playoff_winners = playoff_winners.rename(
    columns={
        "winning team": "team",
        "winning seed": "seed",
        "winning team wins": "series wins"
    }
)

playoff_winners["series won"] = 1

playoff_winners.head()

,season,year,playoff round,team,seed,series wins,series won
0,2025-2026,2026,Eastern Conf First Round,Detroit Pistons,1,4,1
1,2025-2026,2026,Eastern Conf First Round,Cleveland Cavaliers,4,4,1
2,2025-2026,2026,Eastern Conf First Round,New York Knicks,3,4,1
3,2025-2026,2026,Eastern Conf First Round,Philadelphia 76ers,7,4,1
4,2025-2026,2026,Western Conf First Round,Oklahoma City Thunder,1,4,1


In [27]:
playoff_losers = playoff_shrink[
    [
        "season",
        "year",
        "playoff round",
        "losing team",
        "losing seed",
        "losing team wins"
    ]
].copy()

playoff_losers = playoff_losers.rename(
    columns={
        "losing team": "team",
        "losing seed": "seed",
        "losing team wins": "series wins"
    }
)

playoff_losers["series won"] = 0

playoff_losers.head()

,season,year,playoff round,team,seed,series wins,series won
0,2025-2026,2026,Eastern Conf First Round,Orlando Magic,8,3,0
1,2025-2026,2026,Eastern Conf First Round,Toronto Raptors,5,3,0
2,2025-2026,2026,Eastern Conf First Round,Atlanta Hawks,6,2,0
3,2025-2026,2026,Eastern Conf First Round,Boston Celtics,2,3,0
4,2025-2026,2026,Western Conf First Round,Phoenix Suns,8,0,0


In [28]:
playoff_results = pd.concat([playoff_winners, playoff_losers], ignore_index= True)

playoff_results.info()

<class 'pandas.DataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   season         780 non-null    str  
 1   year           780 non-null    int64
 2   playoff round  780 non-null    str  
 3   team           780 non-null    str  
 4   seed           780 non-null    str  
 5   series wins    780 non-null    str  
 6   series won     780 non-null    int64
dtypes: int64(2), str(5)
memory usage: 42.8 KB


In [29]:
playoff_results = playoff_results.drop(columns=["series wins"])

playoff_results["seed"] = playoff_results["seed"].astype(int)
playoff_results["series won"] = playoff_results["series won"].astype(int)

playoff_results

,season,year,playoff round,team,seed,series won
0,2025-2026,2026,Eastern Conf First Round,Detroit Pistons,1,1
1,2025-2026,2026,Eastern Conf First Round,Cleveland Cavaliers,4,1
2,2025-2026,2026,Eastern Conf First Round,New York Knicks,3,1
3,2025-2026,2026,Eastern Conf First Round,Philadelphia 76ers,7,1
4,2025-2026,2026,Western Conf First Round,Oklahoma City Thunder,1,1
...,...,...,...,...,...,...
775,2000-2001,2001,Western Conf Semifinals,Sacramento Kings,3,0
776,2000-2001,2001,Western Conf Semifinals,Dallas Mavericks,5,0
777,2000-2001,2001,Eastern Conf Finals,Milwaukee Bucks,2,0
778,2000-2001,2001,Western Conf Finals,San Antonio Spurs,1,0


In [30]:
playoff_results.info()

<class 'pandas.DataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   season         780 non-null    str  
 1   year           780 non-null    int64
 2   playoff round  780 non-null    str  
 3   team           780 non-null    str  
 4   seed           780 non-null    int64
 5   series won     780 non-null    int64
dtypes: int64(3), str(3)
memory usage: 36.7 KB


In [31]:
playoff_mapping = {
    "Eastern Conf First Round": 1,
    "Western Conf First Round": 1,
    "Eastern Conf Semifinals": 2,
    "Western Conf Semifinals": 2,
    "Eastern Conf Finals": 3,
    "Western Conf Finals": 3,
    "Finals": 4
}

playoff_results["round score"] = playoff_results["playoff round"].map(playoff_mapping)

playoff_results.head()

,season,year,playoff round,team,seed,series won,round score
0,2025-2026,2026,Eastern Conf First Round,Detroit Pistons,1,1,1
1,2025-2026,2026,Eastern Conf First Round,Cleveland Cavaliers,4,1,1
2,2025-2026,2026,Eastern Conf First Round,New York Knicks,3,1,1
3,2025-2026,2026,Eastern Conf First Round,Philadelphia 76ers,7,1,1
4,2025-2026,2026,Western Conf First Round,Oklahoma City Thunder,1,1,1


In [33]:
playoff_results["round score"].value_counts().sort_index()

round score
1    416
2    208
3    104
4     52
Name: count, dtype: int64

In [37]:
playoff_results[["season", "year", "team", "playoff round", "series won", "round score"]].head(20)

,season,year,team,playoff round,series won,round score
0,2025-2026,2026,Detroit Pistons,Eastern Conf First Round,1,1
1,2025-2026,2026,Cleveland Cavaliers,Eastern Conf First Round,1,1
2,2025-2026,2026,New York Knicks,Eastern Conf First Round,1,1
3,2025-2026,2026,Philadelphia 76ers,Eastern Conf First Round,1,1
4,2025-2026,2026,Oklahoma City Thunder,Western Conf First Round,1,1
5,2025-2026,2026,Los Angeles Lakers,Western Conf First Round,1,1
6,2025-2026,2026,Minnesota Timberwolves,Western Conf First Round,1,1
7,2025-2026,2026,San Antonio Spurs,Western Conf First Round,1,1
8,2025-2026,2026,San Antonio Spurs,Western Conf Semifinals,1,2
9,2025-2026,2026,Oklahoma City Thunder,Western Conf Semifinals,1,2


In [57]:
final_postseason_results = playoff_results.groupby(["season", "year", "team"]).agg(
    seed = ("seed", "first"),
    playoff_series_won = ("series won", "sum"),
    round_score = ("round score", "max")
).reset_index()

final_postseason_results.sort_values(by="year", ascending=False).head(20)

,season,year,team,seed,playoff_series_won,round_score
415,2025-2026,2026,Toronto Raptors,5,0,1
407,2025-2026,2026,Minnesota Timberwolves,6,1,2
400,2025-2026,2026,Atlanta Hawks,6,0,1
401,2025-2026,2026,Boston Celtics,2,0,1
402,2025-2026,2026,Cleveland Cavaliers,4,2,3
403,2025-2026,2026,Denver Nuggets,3,0,1
405,2025-2026,2026,Houston Rockets,5,0,1
406,2025-2026,2026,Los Angeles Lakers,4,1,2
404,2025-2026,2026,Detroit Pistons,1,1,2
408,2025-2026,2026,New York Knicks,3,4,4


In [59]:
round_name_map = {
    1: "First Round",
    2: "Conference Semifinals",
    3: "Conference Finals",
    4: "NBA Finals"
}

final_postseason_results["furthest_round"] = (
    final_postseason_results["round_score"].map(round_name_map)
)

final_postseason_results.head(20)

,season,year,team,seed,playoff_series_won,round_score,furthest_round
0,2000-2001,2001,Charlotte Hornets,6,1,2,Conference Semifinals
1,2000-2001,2001,Dallas Mavericks,5,1,2,Conference Semifinals
2,2000-2001,2001,Indiana Pacers,8,0,1,First Round
3,2000-2001,2001,Los Angeles Lakers,2,4,4,NBA Finals
4,2000-2001,2001,Miami Heat,3,0,1,First Round
5,2000-2001,2001,Milwaukee Bucks,2,2,3,Conference Finals
6,2000-2001,2001,Minnesota Timberwolves,8,0,1,First Round
7,2000-2001,2001,New York Knicks,4,0,1,First Round
8,2000-2001,2001,Orlando Magic,7,0,1,First Round
9,2000-2001,2001,Philadelphia 76ers,1,3,4,NBA Finals


In [52]:
final_postseason_results.shape

(416, 4)

In [60]:
final_postseason_results.duplicated(
    subset=["season", "team"]
).sum()

np.int64(0)

In [61]:
final_postseason_results.columns

Index(['season', 'year', 'team', 'seed', 'playoff_series_won', 'round_score',
       'furthest_round'],
      dtype='str')

In [63]:
final_postseason_results.groupby(
    ["season", "playoff_series_won"]
).size()

season     playoff_series_won
2000-2001  0                     8
           1                     4
           2                     2
           3                     1
           4                     1
                                ..
2025-2026  0                     8
           1                     4
           2                     2
           3                     1
           4                     1
Length: 130, dtype: int64

In [64]:
final_postseason_results.isna().sum()

season                0
year                  0
team                  0
seed                  0
playoff_series_won    0
round_score           0
furthest_round        0
dtype: int64

In [66]:
import os

os.makedirs("../data/staging/postseason", exist_ok=True)

playoff_results.to_csv(
    "../data/staging/postseason/s03_playoff_results.csv",
    index=False
)

final_postseason_results.to_csv(
    "../data/staging/postseason/s03_team_season_postseason.csv",
    index=False
)